In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Dict

In [32]:
class PositionalEncoding(nn.Module):
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        super().__init__()

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        self.num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(self.num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(self.num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [34]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [35]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim 
        self.attention_layer = MultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm2(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [36]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
hidden_dim = 16

In [37]:
dummy_block_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_block_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_block_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.6527, 0.7673, 0.2262, 0.9859, 0.1074, 0.1218, 0.5954, 0.8205],
         [0.8108, 0.4809, 0.8490, 0.3781, 0.7153, 0.1835, 0.4544, 0.1960],
         [0.2714, 0.2062, 0.2394, 0.8944, 0.7676, 0.5183, 0.3941, 0.8289]],

        [[0.2727, 0.2178, 0.7810, 0.6065, 0.5450, 0.4291, 0.6675, 0.7915],
         [0.6637, 0.9994, 0.7341, 0.1461, 0.4629, 0.3863, 0.6903, 0.4849],
         [0.1023, 0.0520, 0.8861, 0.8079, 0.5033, 0.1594, 0.0266, 0.2547]]])

In [38]:
encoder = EncoderBlock(d_model, num_heads, hidden_dim)
encoder

EncoderBlock(
  (attention_layer): MultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=True)
    (w_k): Linear(in_features=8, out_features=8, bias=True)
    (w_v): Linear(in_features=8, out_features=8, bias=True)
    (w_o): Linear(in_features=8, out_features=8, bias=True)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [39]:
dummy_output = encoder(dummy_block_input)
dummy_output

tensor([[[ 0.7049,  0.6651,  0.5548,  0.5445,  0.2981,  0.1112,  0.5562,
           0.9995],
         [ 0.9195,  0.3611,  1.1302, -0.2545,  0.7629, -0.0075,  0.2728,
           0.6324],
         [ 0.0500, -0.0273,  0.5849,  0.1998,  0.6598,  0.6216,  0.4105,
           1.0966]],

        [[ 0.1034, -0.2083,  1.1024,  0.0330,  0.1011,  0.4957,  0.5873,
           1.5160],
         [ 0.8068,  0.5793,  1.1374, -0.2961,  0.0635,  0.8157,  0.6620,
           1.0467],
         [-0.0878, -0.5033,  0.6154,  0.0861, -0.0800,  0.2338, -0.0406,
           0.8912]]], grad_fn=<AddBackward0>)

In [40]:
# Now, Stack Multiple Encoders:
class EncoderStack(nn.Module):
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding()
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, hidden_dim) for _ in range(6)])
        self.norm_1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)
        
    def forward(self, tokens: torch.Tensor) -> torch.Tensor: 
        # Dimension of tokens: (batch_size, num_tokens) 

        embedded = self.embedding(tokens) 
        # Dimension: (batch_size, batch_size, d_model)

        embedded_scaled = embedded * math.sqrt(self.d_model)
        positional_embedding = self.positional_encoding(embedded_scaled)
        x = self.dropout(positional_embedding)

        for layer in self.layers:
            x = layer(x)
        output = self.norm_1(x)

        return output

In [41]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 5
hidden_dim = 16

In [42]:
dummy_stack_input = torch.randint(low=0, high=vocab_size, size=(batch_size, num_tokens))
print(dummy_stack_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_stack_input

torch.Size([2, 3])
(batch_size, num_tokens, embedding_dimension)


tensor([[3, 1, 4],
        [4, 2, 0]])

In [43]:
encoder_stack = EncoderStack(vocab_size, d_model, num_heads, hidden_dim)
encoder_stack

EncoderStack(
  (embedding): Embedding(5, 8)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-5): 6 x EncoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=True)
        (w_k): Linear(in_features=8, out_features=8, bias=True)
        (w_v): Linear(in_features=8, out_features=8, bias=True)
        (w_o): Linear(in_features=8, out_features=8, bias=True)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm_1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=Fa

In [44]:
encoder_stack_output = encoder_stack(dummy_stack_input)
print(encoder_stack_output.shape)
encoder_stack_output

torch.Size([2, 3, 8])


tensor([[[ 0.2047,  0.3226, -0.9461,  1.4552,  1.2865,  0.0831, -0.8258,
          -1.5802],
         [ 0.9096, -0.2516, -1.6910, -0.4350,  0.0086,  1.5146, -0.9671,
           0.9118],
         [ 1.8573, -0.6756, -0.9979,  0.4566, -1.2365,  0.5901, -0.7086,
           0.7145]],

        [[-0.8026,  0.5482, -1.0162,  0.5288, -1.3088,  0.8117, -0.5216,
           1.7604],
         [-1.3617,  0.0149,  1.2098,  0.1247, -1.6052, -0.1548,  1.3838,
           0.3886],
         [-1.1304, -0.1861, -0.5727,  2.3083, -0.2565,  0.1982, -0.8366,
           0.4757]]], grad_fn=<NativeLayerNormBackward0>)